# Fine-tuning y evaluación de modelos de lenguaje
## Clasificación de emociones en texto con DistilBERT


---
# Bloque 0. Configuración del entorno

**Instrucciones**

1. Abrir este notebook en **Google Colab**.
2. Ir a `Entorno de ejecución > Cambiar tipo de entorno de ejecución` y seleccionar **GPU T4**.
3. Ejecutar las celdas de esta sección.

**Por qué:** el ajuste fino multiplica millones de parámetros en cada paso. En una GPU, una época con 4.000 frases tarda menos de un minuto; en CPU puede tardar más de 20 minutos. El código funciona en ambos casos, solo cambia el tiempo.

In [ ]:
# Instalación de librerías (en Colab la mayoría ya vienen instaladas)
# Este notebook usa la API de transformers 5.x
!pip install -q "transformers>=5.0" datasets accelerate scikit-learn

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline,
    set_seed,
)
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)

SEMILLA = 42
set_seed(SEMILLA)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Versión de transformers:", transformers.__version__)
print("Dispositivo:", DISPOSITIVO)
if DISPOSITIVO == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

**Por qué fijamos una semilla:** el ajuste fino tiene componentes aleatorios (inicialización de la cabeza de clasificación, orden de los lotes, dropout). Fijar la semilla hace que los resultados sean **reproducibles**, condición indispensable para comparar experimentos en el Bloque 7.

---
# Bloque 1. Del Transformer al modelo útil (15 min)

## 1.1 Dos etapas en la vida de un modelo de lenguaje

| | Preentrenamiento | Ajuste fino (fine-tuning) |
|---|---|---|
| **Objetivo** | Aprender cómo funciona el lenguaje | Resolver una tarea concreta |
| **Datos** | Enormes cantidades de texto **sin etiquetas** | Pocos miles de ejemplos **etiquetados** |
| **Tarea** | Predecir palabras ocultas (*masked language modeling*) | Clasificar, extraer, responder... |
| **Costo** | Semanas en cientos de GPU | Minutos u horas en una GPU |
| **Quién lo hace** | Grandes laboratorios | Cualquier equipo de desarrollo |

La idea clave es la **transferencia de aprendizaje**: reutilizamos todo lo que el modelo aprendió sobre el lenguaje y solo le enseñamos la tarea final.

## 1.2 La cabeza de clasificación

En la clase anterior vimos el **encoder** del Transformer: recibe tokens y devuelve un vector contextual por cada token. Para clasificar una frase completa necesitamos **un solo vector** que la represente. Por eso los modelos tipo BERT agregan un token especial al inicio, `[CLS]`, cuyo vector final resume la frase.

Sobre ese vector se coloca una **cabeza de clasificación**: una pequeña red densa que transforma el vector de 768 dimensiones en 6 valores (uno por emoción), llamados **logits**. La función softmax convierte los logits en probabilidades:

$$P(y = k \mid x) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

donde $z_k$ es el logit de la clase $k$ y $K = 6$ es el número de emociones.

```
"I feel so happy today"
        |
   Tokenizador  ->  [CLS] i feel so happy today [SEP]
        |
   Encoder DistilBERT (6 capas, preentrenado)
        |
   Vector de [CLS] (768 dimensiones)
        |
   Cabeza de clasificación (NUEVA, pesos aleatorios)
        |
   6 logits -> softmax -> probabilidades por emoción
```

## 1.3 El problema: un modelo que aún no sabe la tarea

Vamos a cargar DistilBERT con una cabeza de clasificación de 6 clases **sin entrenarla** y le pediremos clasificar frases obvias.

In [ ]:
MODELO_BASE = "distilbert-base-uncased"

# Etiquetas del dataset dair-ai/emotion, en el mismo orden que usa el dataset
ETIQUETAS_EN = ["sadness", "joy", "love", "anger", "fear", "surprise"]
ETIQUETAS_ES = ["tristeza", "alegría", "amor", "ira", "miedo", "sorpresa"]

id2label = {i: e for i, e in enumerate(ETIQUETAS_ES)}
label2id = {e: i for i, e in enumerate(ETIQUETAS_ES)}

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
modelo_sin_ajuste = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=len(ETIQUETAS_ES),
    id2label=id2label,
    label2id=label2id,
).to(DISPOSITIVO)

**Observe la advertencia** que aparece al cargar el modelo: indica que algunos pesos (`classifier`, `pre_classifier`) **fueron inicializados aleatoriamente**. Esa es exactamente la cabeza de clasificación nueva. La librería nos está avisando que el modelo debe entrenarse antes de usarse.

In [ ]:
def predecir(modelo, frases, top=2):
    # Devuelve las 'top' emociones más probables para cada frase
    modelo.eval()
    entradas = tokenizer(frases, padding=True, truncation=True,
                         max_length=64, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        logits = modelo(**entradas).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    filas = []
    for frase, p in zip(frases, probs):
        orden = np.argsort(p)[::-1][:top]
        fila = {"frase": frase}
        for r, idx in enumerate(orden, start=1):
            fila[f"emoción {r}"] = modelo.config.id2label[int(idx)]
            fila[f"prob {r}"] = round(float(p[idx]), 3)
        filas.append(fila)
    return pd.DataFrame(filas)

FRASES_DEMO = [
    "I feel so happy and grateful today",
    "I am terrified of what might happen tomorrow",
    "I miss my grandmother so much, I can't stop crying",
    "I am furious, they lied to me again",
    "I love spending time with you",
    "I can't believe it, I won the lottery!",
]

predecir(modelo_sin_ajuste, FRASES_DEMO)

**Pregunta para la clase:** ¿las probabilidades son razonables?

Las probabilidades rondan $1/6 \approx 0{,}17$ para todas las clases y las predicciones no tienen relación con el significado de la frase. El encoder sí "entiende" el texto, pero la cabeza de clasificación todavía no sabe qué emoción corresponde a cada representación.

**Conclusión del bloque:** necesitamos **ajustar** el modelo con ejemplos etiquetados. Ese es el objetivo de esta clase.

In [ ]:
# Liberamos memoria: este modelo solo servía para la demostración
del modelo_sin_ajuste
if DISPOSITIVO == "cuda":
    torch.cuda.empty_cache()

---
# Bloque 2. Anatomía del modelo y sus parámetros (25 min)

**Instrucciones:** ejecute cada celda y relacione lo que ve con la arquitectura Transformer de la clase anterior.

**Por qué:** antes de entrenar un modelo hay que saber **qué** se va a entrenar. El número de parámetros determina el costo en memoria y tiempo, y saber dónde está cada capa permite decidir qué partes congelar.

In [ ]:
modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=len(ETIQUETAS_ES),
    id2label=id2label,
    label2id=label2id,
).to(DISPOSITIVO)

print(modelo)

### Lectura guiada de la arquitectura

| Componente | Qué es | Relación con la clase anterior |
|---|---|---|
| `embeddings.word_embeddings` | Tabla de 30.522 tokens x 768 dimensiones | Vectores iniciales de cada token |
| `embeddings.position_embeddings` | 512 posiciones x 768 dimensiones | Codificación posicional |
| `transformer.layer[0..5]` | 6 bloques Transformer | Atención multi-cabeza + red feed-forward |
| `attention.q_lin`, `k_lin`, `v_lin` | Proyecciones lineales | Matrices $W_Q$, $W_K$, $W_V$ |
| `ffn.lin1`, `ffn.lin2` | Red feed-forward (768 -> 3072 -> 768) | Capa posición a posición |
| `pre_classifier`, `classifier` | **Cabeza nueva** (768 -> 768 -> 6) | Lo que agregamos para la tarea |

DistilBERT es una versión **destilada** de BERT: conserva cerca del 97% de su desempeño con 6 capas en lugar de 12, por eso es ideal para clase.

In [ ]:
def contar_parametros(modelo):
    total = sum(p.numel() for p in modelo.parameters())
    entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    return total, entrenables

total, entrenables = contar_parametros(modelo)
print(f"Parámetros totales:     {total:>12,}")
print(f"Parámetros entrenables: {entrenables:>12,}")

# Parámetros por componente
componentes = {
    "Embeddings": modelo.base_model.embeddings,
    "Capas Transformer (6)": modelo.base_model.transformer,
    "Cabeza de clasificación": torch.nn.ModuleList([modelo.pre_classifier, modelo.classifier]),
}
resumen = pd.DataFrame([
    {"componente": n, "parámetros": sum(p.numel() for p in m.parameters())}
    for n, m in componentes.items()
])
resumen["porcentaje"] = (100 * resumen["parámetros"] / total).round(2)
resumen

**Pregunta para la clase:** la cabeza de clasificación representa menos del 1% del modelo. ¿Qué implica entrenar solo esa parte frente a entrenar todo el modelo?

## 2.1 El tokenizador

El modelo no lee texto, lee números. El tokenizador divide la frase en **subpalabras** (algoritmo WordPiece) y las convierte en identificadores.

In [ ]:
frase = "I am unbelievably happy with the results"
codificado = tokenizer(frase)

print("Tokens:        ", tokenizer.convert_ids_to_tokens(codificado["input_ids"]))
print("input_ids:     ", codificado["input_ids"])
print("attention_mask:", codificado["attention_mask"])

Observe:

- `[CLS]` (id 101) al inicio y `[SEP]` (id 102) al final se agregan automáticamente.
- Palabras poco frecuentes se dividen en subpalabras con el prefijo `##` (por ejemplo `unbelievably`). Así el vocabulario se mantiene en unos 30.000 tokens sin palabras desconocidas.
- `attention_mask` vale 1 en tokens reales y 0 en tokens de relleno (*padding*). Le indica al mecanismo de atención qué posiciones ignorar.

## 2.2 Tres estrategias de ajuste

| Estrategia | Qué se entrena | Ventaja | Desventaja |
|---|---|---|---|
| **Extracción de características** (encoder congelado) | Solo la cabeza (~0,6 M parámetros) | Muy rápido, poca memoria | El encoder no se adapta al dominio |
| **Ajuste fino completo** | Todo el modelo (~67 M parámetros) | Mejor desempeño | Más costo, riesgo de olvido catastrófico |
| **PEFT / LoRA** | Pequeñas matrices añadidas a las capas de atención | Casi el desempeño del ajuste completo con ~1% de los parámetros | Más complejo de configurar |

En esta clase compararemos las dos primeras. LoRA es la estrategia estándar para modelos grandes (LLM de miles de millones de parámetros), donde el ajuste completo es inviable.

In [ ]:
def congelar_encoder(modelo):
    # Congela todo el encoder: solo la cabeza de clasificación queda entrenable
    for p in modelo.base_model.parameters():
        p.requires_grad = False
    return modelo

# Demostración con una copia temporal
demo = AutoModelForSequenceClassification.from_pretrained(MODELO_BASE, num_labels=6)
demo = congelar_encoder(demo)
t, e = contar_parametros(demo)
print(f"Encoder congelado -> entrenables: {e:,} de {t:,} ({100*e/t:.2f} %)")
del demo

---
# Bloque 3. Preparación de datos (20 min)

**Dataset:** [`dair-ai/emotion`](https://huggingface.co/datasets/dair-ai/emotion), frases en inglés tomadas de Twitter y etiquetadas con 6 emociones básicas.

**Por qué este dataset:** está limpio, documentado y presenta **desbalance de clases**, situación muy frecuente en proyectos reales y que nos obligará a elegir bien la métrica de evaluación.

In [ ]:
dataset = load_dataset("dair-ai/emotion")
dataset

In [ ]:
# Verificamos que el orden de las etiquetas coincide con el que definimos
print(dataset["train"].features["label"].names)
assert dataset["train"].features["label"].names == ETIQUETAS_EN

In [ ]:
df_train = dataset["train"].to_pandas()
df_train["emoción"] = df_train["label"].map(id2label)
df_train.sample(8, random_state=SEMILLA)

In [ ]:
conteo = df_train["emoción"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
conteo.plot(kind="barh", ax=ax, color="#4C72B0")
ax.invert_yaxis()
ax.set_title("Distribución de emociones en el conjunto de entrenamiento")
ax.set_xlabel("Número de frases")
for i, v in enumerate(conteo.values):
    ax.text(v + 50, i, f"{v} ({100*v/len(df_train):.1f}%)", va="center")
plt.tight_layout()
plt.show()

**Pregunta para la clase:** si un modelo respondiera siempre "alegría", ¿qué accuracy obtendría? (Aproximadamente un tercio). Por eso la accuracy sola es engañosa con clases desbalanceadas.

## 3.1 Longitud de las frases

Necesitamos decidir el `max_length` del tokenizador: frases más largas se truncan.

In [ ]:
longitudes = [len(tokenizer(t)["input_ids"]) for t in df_train["text"].sample(2000, random_state=SEMILLA)]
print(f"Percentil 95 de longitud en tokens: {np.percentile(longitudes, 95):.0f}")
print(f"Máximo: {max(longitudes)}")

plt.figure(figsize=(8, 3))
plt.hist(longitudes, bins=40, color="#55A868")
plt.axvline(64, color="red", linestyle="--", label="max_length = 64")
plt.xlabel("Tokens por frase"); plt.ylabel("Frecuencia"); plt.legend()
plt.show()

Con `max_length = 64` cubrimos prácticamente todas las frases. Como el costo de la atención crece con el **cuadrado** de la longitud, una longitud corta acelera mucho el entrenamiento.

## 3.2 Submuestra y particiones

**Instrucciones:** usaremos 4.000 frases de entrenamiento, el conjunto de validación completo para monitorear el entrenamiento y el conjunto de prueba completo **solo al final**.

**Por qué:** 4.000 ejemplos bastan para un buen resultado en pocos minutos y permiten repetir experimentos en clase. El conjunto de prueba no se usa para tomar decisiones; si se usara, la evaluación final estaría sesgada.

In [ ]:
N_TRAIN = 4000   # puede subir a 16000 (dataset completo) si dispone de tiempo

train_ds = dataset["train"].shuffle(seed=SEMILLA).select(range(N_TRAIN))
val_ds = dataset["validation"]
test_ds = dataset["test"]

print(f"Entrenamiento: {len(train_ds):,} | Validación: {len(val_ds):,} | Prueba: {len(test_ds):,}")

In [ ]:
MAX_LEN = 64

def tokenizar(lote):
    return tokenizer(lote["text"], truncation=True, max_length=MAX_LEN)

train_tok = train_ds.map(tokenizar, batched=True)
val_tok = val_ds.map(tokenizar, batched=True)
test_tok = test_ds.map(tokenizar, batched=True)

# Padding dinámico: cada lote se rellena solo hasta la frase más larga del lote
collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_tok

**Por qué padding dinámico:** si rellenáramos todas las frases a 64 tokens, la mayoría de los cálculos se harían sobre ceros. `DataCollatorWithPadding` rellena cada lote solo hasta la frase más larga de ese lote, lo que ahorra tiempo sin cambiar el resultado.

---
# Bloque 4. Ajuste fino (40 min)

## 4.1 Qué ocurre durante el entrenamiento

En cada paso:

1. Se toma un **lote** de frases y se calculan las predicciones (paso hacia adelante).
2. Se mide el error con la **entropía cruzada**:
$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \log P(y_i \mid x_i)$$
   donde $y_i$ es la emoción correcta de la frase $x_i$. Si el modelo asigna probabilidad alta a la clase correcta, la pérdida es baja.
3. Se calculan los gradientes (retropropagación) y el optimizador **AdamW** actualiza los pesos:
$$\theta \leftarrow \theta - \eta \cdot \text{paso}_{\text{AdamW}}(\nabla_\theta \mathcal{L})$$
   donde $\eta$ es la **tasa de aprendizaje** (*learning rate*).

## 4.2 Hiperparámetros

| Hiperparámetro | Valor | Por qué |
|---|---|---|
| `learning_rate` | 2e-5 | Muy bajo a propósito: pasos grandes destruyen lo aprendido en el preentrenamiento (**olvido catastrófico**) |
| `per_device_train_batch_size` | 16 | Frases procesadas por paso; equilibrio entre memoria de GPU y estabilidad del gradiente |
| `num_train_epochs` | 3 | Veces que el modelo recorre todo el conjunto; con 2-4 suele bastar en ajuste fino |
| `weight_decay` | 0.01 | Regularización: penaliza pesos grandes para reducir sobreajuste |
| `warmup_steps` | 0.1 | Un valor entre 0 y 1 se interpreta como proporción: el 10% inicial de los pasos sube la tasa de aprendizaje gradualmente, evitando cambios bruscos cuando la cabeza aún es aleatoria |

**Nota técnica (transformers 5.x):** muchos tutoriales antiguos usan argumentos que ya no existen y producen error:

- `evaluation_strategy` ahora se llama `eval_strategy`.
- `warmup_ratio` se eliminó: se usa `warmup_steps` con un valor decimal entre 0 y 1.
- `Trainer(tokenizer=...)` ahora es `Trainer(processing_class=...)`.

In [ ]:
def calcular_metricas(eval_pred):
    logits, etiquetas = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(etiquetas, preds),
        "f1_macro": f1_score(etiquetas, preds, average="macro"),
    }

args = TrainingArguments(
    output_dir="salida_emociones",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=0.1,
    eval_strategy="epoch",       # evalúa en validación al final de cada época
    save_strategy="no",          # guardaremos el modelo manualmente al final
    logging_steps=25,            # registra la pérdida de entrenamiento cada 25 pasos
    fp16=(DISPOSITIVO == "cuda"),  # precisión mixta: más rápido en GPU
    report_to="none",            # sin servicios externos de seguimiento
    seed=SEMILLA,
)

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
    processing_class=tokenizer,
    compute_metrics=calcular_metricas,
)

pasos_por_epoca = int(np.ceil(N_TRAIN / args.per_device_train_batch_size))
print(f"Pasos por época: {pasos_por_epoca} | Pasos totales: {pasos_por_epoca * int(args.num_train_epochs)}")

**Instrucciones:** ejecute la siguiente celda. Mientras entrena (1-3 minutos en GPU), observe cómo baja la pérdida y discuta con su grupo: ¿qué pasaría con una tasa de aprendizaje 100 veces mayor?

In [ ]:
resultado = trainer.train()
print(resultado)

## 4.3 Curvas de aprendizaje

**Por qué:** las curvas son el diagnóstico principal del entrenamiento.

- Si la pérdida de **entrenamiento** baja y la de **validación** también: el modelo aprende y generaliza.
- Si la de entrenamiento sigue bajando pero la de validación **sube**: **sobreajuste**, el modelo memoriza en lugar de aprender.

In [ ]:
historial = pd.DataFrame(trainer.state.log_history)
perd_train = historial.dropna(subset=["loss"])[["epoch", "loss"]]
perd_val = historial.dropna(subset=["eval_loss"])[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(perd_train["epoch"], perd_train["loss"], label="Entrenamiento", alpha=0.8)
axes[0].plot(perd_val["epoch"], perd_val["eval_loss"], "o-", label="Validación", linewidth=2)
axes[0].set_title("Pérdida (entropía cruzada)"); axes[0].set_xlabel("Época"); axes[0].legend()

axes[1].plot(perd_val["epoch"], perd_val["eval_accuracy"], "o-", label="Accuracy")
axes[1].plot(perd_val["epoch"], perd_val["eval_f1_macro"], "s-", label="F1-macro")
axes[1].set_title("Métricas en validación"); axes[1].set_xlabel("Época"); axes[1].set_ylim(0, 1); axes[1].legend()
plt.tight_layout(); plt.show()

perd_val.round(4)

## 4.4 Guardar el modelo ajustado

**Por qué:** un modelo ajustado es un artefacto reutilizable. Se guardan los pesos y el tokenizador juntos, porque el modelo solo funciona con el mismo tokenizador con el que fue entrenado.

In [ ]:
RUTA_MODELO = "modelo_emociones_distilbert"
trainer.save_model(RUTA_MODELO)   # guarda pesos, configuración y tokenizador

# Opcional (Colab): copiar a Google Drive para conservarlo
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r modelo_emociones_distilbert /content/drive/MyDrive/

---
# Receso (10 min)
---

# Bloque 5. Probar el modelo (20 min)

**Probar no es lo mismo que evaluar.** Probar es interactuar con el modelo para formar una intuición cualitativa de cómo se comporta. Evaluar (Bloque 6) es medir de forma sistemática con datos que el modelo nunca vio.

## 5.1 Las mismas frases del inicio

In [ ]:
predecir(trainer.model, FRASES_DEMO)

**Compare** con la tabla del Bloque 1: con el mismo encoder y la misma arquitectura, unos minutos de ajuste fino cambian completamente el comportamiento.

## 5.2 Uso como pipeline

En producción se usa la interfaz `pipeline`, que encapsula tokenización, inferencia y softmax.

In [ ]:
clasificador = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0 if DISPOSITIVO == "cuda" else -1,
)
clasificador("I just got accepted into my dream university")

## 5.3 Casos difíciles

**Instrucciones:** cada grupo agrega al menos 3 frases propias a la lista de abajo, intentando "engañar" al modelo. Anote cuáles falla y proponga una explicación.

In [ ]:
FRASES_DIFICILES = [
    # Negaciones
    "I am not sad at all",
    "I don't hate you",
    # Sarcasmo
    "Oh great, another Monday. I'm just thrilled",
    # Ambigüedad
    "I can't believe what just happened",
    # Emoción mixta
    "I'm happy for her but I will miss her terribly",
    # Frases de los estudiantes:
    # "...",
]
predecir(trainer.model, FRASES_DIFICILES, top=3)

## 5.4 ¿Y en español?

In [ ]:
FRASES_ES = [
    "Estoy muy feliz con mis resultados",
    "Tengo mucho miedo de lo que pueda pasar",
    "Estoy furioso, me volvieron a mentir",
    "Te quiero muchísimo",
]
predecir(trainer.model, FRASES_ES, top=3)

**Discusión:** el modelo falla o acierta por casualidad. Razones:

1. `distilbert-base-uncased` fue **preentrenado solo en inglés**: su vocabulario parte las palabras en español en fragmentos sin significado.
2. Los datos de ajuste fino también están en inglés.

**¿Qué necesitaríamos para el español?**

- Un modelo base en español, por ejemplo `dccuchile/bert-base-spanish-wwm-cased` (BETO) o `PlanTL-GOB-ES/roberta-base-bne`, o uno multilingüe como `xlm-roberta-base`.
- Datos etiquetados en español.

**Lección general:** el ajuste fino adapta el modelo a una **tarea**, pero no le da conocimientos que no adquirió en el preentrenamiento.

---
# Bloque 6. Evaluación formal (35 min)

**Instrucciones:** a partir de aquí usamos el **conjunto de prueba** (2.000 frases que el modelo nunca vio ni se usaron para decidir nada).

## 6.1 Métricas

Para cada clase $k$:

$$\text{Precisión}_k = \frac{VP_k}{VP_k + FP_k} \qquad \text{Recall}_k = \frac{VP_k}{VP_k + FN_k} \qquad F1_k = 2\cdot\frac{\text{Precisión}_k \cdot \text{Recall}_k}{\text{Precisión}_k + \text{Recall}_k}$$

- **Precisión:** de las veces que el modelo dijo "miedo", ¿cuántas era realmente miedo?
- **Recall (exhaustividad):** de todas las frases de miedo, ¿cuántas detectó?
- **F1-macro:** promedio simple del F1 de las 6 clases:
$$F1_{\text{macro}} = \frac{1}{K}\sum_{k=1}^{K} F1_k$$

**Por qué F1-macro:** da el mismo peso a todas las clases. Una clase minoritaria como "sorpresa" pesa igual que "alegría", así que el modelo no puede obtener buena nota ignorando las clases pequeñas.

In [ ]:
pred_test = trainer.predict(test_tok)
y_true = pred_test.label_ids
y_pred = np.argmax(pred_test.predictions, axis=-1)

print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"F1-macro: {f1_score(y_true, y_pred, average='macro'):.4f}\n")
print(classification_report(y_true, y_pred, target_names=ETIQUETAS_ES, digits=3))

**Pregunta para la clase:** ¿qué clases tienen el F1 más bajo? ¿Coinciden con las clases minoritarias del gráfico del Bloque 3?

## 6.2 Matriz de confusión

**Por qué:** las métricas dicen **cuánto** se equivoca el modelo; la matriz de confusión dice **con qué** se confunde. Está normalizada por fila: cada fila suma 1 y muestra hacia dónde van las frases de cada emoción real.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred,
    display_labels=ETIQUETAS_ES,
    normalize="true",
    values_format=".2f",
    cmap="Blues",
    ax=ax,
)
ax.set_title("Matriz de confusión normalizada (conjunto de prueba)")
ax.set_xlabel("Predicción"); ax.set_ylabel("Emoción real")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

Confusiones típicas en este dataset: **amor con alegría** y **sorpresa con miedo**. Ambas parejas comparten vocabulario y, en algunos casos, incluso a una persona le costaría decidir.

## 6.3 Análisis de errores

**Instrucciones:** revise los errores en los que el modelo estaba **más seguro**. Para cada uno, decida: ¿se equivocó el modelo o la etiqueta original es discutible?

**Por qué:** el análisis de errores es la fuente principal de ideas para mejorar un modelo. A veces revela problemas en los datos que ninguna métrica muestra.

In [ ]:
probs_test = torch.softmax(torch.tensor(pred_test.predictions), dim=-1).numpy()

errores = pd.DataFrame({
    "frase": test_ds["text"],
    "real": [id2label[i] for i in y_true],
    "predicha": [id2label[i] for i in y_pred],
    "confianza": probs_test.max(axis=1).round(3),
})
errores = errores[errores["real"] != errores["predicha"]].sort_values("confianza", ascending=False)

print(f"Total de errores: {len(errores)} de {len(test_ds)}")
pd.set_option("display.max_colwidth", 120)
errores.head(15)

## 6.4 Comparación con líneas base

**Por qué:** un 90% aislado no dice nada. Necesitamos saber cuánto mejor es el ajuste fino frente a alternativas más simples y baratas. Comparamos tres enfoques con **los mismos datos de entrenamiento y prueba**:

1. **TF-IDF + regresión logística:** el enfoque clásico que vimos en la clase de embeddings.
2. **DistilBERT con encoder congelado:** solo se entrena la cabeza.
3. **DistilBERT con ajuste fino completo:** el modelo que acabamos de entrenar.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizador = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train = vectorizador.fit_transform(train_ds["text"])
X_test = vectorizador.transform(test_ds["text"])

regresion = LogisticRegression(max_iter=2000, C=5)
regresion.fit(X_train, train_ds["label"])
pred_tfidf = regresion.predict(X_test)

resultados = [{
    "enfoque": "TF-IDF + regresión logística",
    "parámetros entrenables": X_train.shape[1] * 6 + 6,
    "accuracy": accuracy_score(y_true, pred_tfidf),
    "f1_macro": f1_score(y_true, pred_tfidf, average="macro"),
}]
pd.DataFrame(resultados).round(4)

Ahora la segunda línea base. Como solo se entrena la cabeza, usamos una tasa de aprendizaje mayor (1e-3): no hay pesos preentrenados que proteger.

In [ ]:
def entrenar_y_evaluar(nombre, learning_rate=2e-5, epocas=3, congelar=False, mostrar=True):
    # Entrena un modelo nuevo desde DistilBERT preentrenado y lo evalúa en prueba
    set_seed(SEMILLA)
    m = AutoModelForSequenceClassification.from_pretrained(
        MODELO_BASE, num_labels=6, id2label=id2label, label2id=label2id
    )
    if congelar:
        congelar_encoder(m)
    _, n_entrenables = contar_parametros(m)

    a = TrainingArguments(
        output_dir=f"salida_{nombre}",
        learning_rate=learning_rate,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        num_train_epochs=epocas,
        weight_decay=0.01,
        warmup_steps=0.1,
        eval_strategy="no",
        save_strategy="no",
        logging_strategy="no",
        fp16=(DISPOSITIVO == "cuda"),
        report_to="none",
        seed=SEMILLA,
    )
    t = Trainer(model=m, args=a, train_dataset=train_tok,
                data_collator=collator, processing_class=tokenizer)
    t.train()
    p = np.argmax(t.predict(test_tok).predictions, axis=-1)

    fila = {
        "enfoque": nombre,
        "parámetros entrenables": n_entrenables,
        "accuracy": accuracy_score(y_true, p),
        "f1_macro": f1_score(y_true, p, average="macro"),
    }
    del m, t
    if DISPOSITIVO == "cuda":
        torch.cuda.empty_cache()
    if mostrar:
        print(f"{nombre}: F1-macro = {fila['f1_macro']:.4f}")
    return fila

resultados.append(entrenar_y_evaluar("DistilBERT congelado (solo cabeza)",
                                     learning_rate=1e-3, epocas=3, congelar=True))

In [ ]:
_, n_completo = contar_parametros(trainer.model)
resultados.append({
    "enfoque": "DistilBERT ajuste fino completo",
    "parámetros entrenables": n_completo,
    "accuracy": accuracy_score(y_true, y_pred),
    "f1_macro": f1_score(y_true, y_pred, average="macro"),
})

tabla = pd.DataFrame(resultados)
tabla["parámetros entrenables"] = tabla["parámetros entrenables"].map("{:,}".format)
tabla.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(tabla["enfoque"], tabla["f1_macro"], color=["#8C8C8C", "#DD8452", "#4C72B0"])
for i, v in enumerate(tabla["f1_macro"]):
    ax.text(v + 0.01, i, f"{v:.3f}", va="center")
ax.set_xlim(0, 1); ax.set_xlabel("F1-macro en prueba")
ax.set_title("Comparación de enfoques (mismos datos de entrenamiento y prueba)")
ax.invert_yaxis(); plt.tight_layout(); plt.show()

**Preguntas para la clase:**

1. El encoder congelado, ¿supera a TF-IDF? ¿Por qué podría no hacerlo, si DistilBERT "entiende" mejor el lenguaje?
2. ¿Cuánto mejora el ajuste fino completo frente a la mejor línea base? ¿Justifica el costo adicional en parámetros y tiempo?
3. Si tuviera que desplegar este clasificador en un servidor sin GPU, ¿cuál elegiría y por qué?

**Idea clave:** los vectores de `[CLS]` de un modelo preentrenado no están optimizados para emociones. El ajuste fino completo **reorganiza** las representaciones internas para que las emociones queden separadas, y eso no se logra entrenando solo la cabeza.

---
# Bloque 7. Experimento por grupos y cierre (15 min)

**Instrucciones:**

1. Cada grupo ejecuta **un solo** experimento de la tabla, cambiando **una sola** variable respecto a la configuración base (`learning_rate=2e-5`, `epocas=3`, ajuste completo).
2. Anote su F1-macro en la tabla compartida del curso.
3. Prepare una explicación de 1 minuto del resultado.

| Grupo | Variable modificada | Llamada |
|---|---|---|
| 1 | Tasa de aprendizaje alta | `entrenar_y_evaluar("lr_5e-5", learning_rate=5e-5)` |
| 2 | Tasa de aprendizaje muy alta | `entrenar_y_evaluar("lr_1e-4", learning_rate=1e-4)` |
| 3 | Tasa de aprendizaje extrema | `entrenar_y_evaluar("lr_1e-3", learning_rate=1e-3)` |
| 4 | Una sola época | `entrenar_y_evaluar("1_epoca", epocas=1)` |
| 5 | Más épocas | `entrenar_y_evaluar("6_epocas", epocas=6)` |

**Por qué una variable a la vez:** si se cambian dos hiperparámetros a la vez y el resultado mejora, no se puede saber cuál produjo el efecto. Es el mismo principio del diseño experimental.

In [ ]:
# Ejecute SOLO la línea de su grupo
experimento = entrenar_y_evaluar("lr_5e-5", learning_rate=5e-5)
experimento

### Resultados esperados (para la discusión)

- **Tasa alta (1e-3):** el F1 suele colapsar; el modelo puede terminar prediciendo una sola clase. Es el **olvido catastrófico** en acción.
- **Una época:** desempeño algo menor, el modelo aún no converge.
- **Seis épocas:** mejora poco o nada; en validación puede aparecer sobreajuste.

## Cierre: síntesis de la clase

1. El **preentrenamiento** aporta conocimiento general del lenguaje; el **ajuste fino** lo adapta a una tarea con pocos datos.
2. Un modelo preentrenado con cabeza nueva **no sirve** sin ajuste: la cabeza empieza aleatoria.
3. La **tasa de aprendizaje** es el hiperparámetro más delicado del ajuste fino.
4. **Probar** forma intuición; **evaluar** mide. Ambas cosas son necesarias.
5. Con clases desbalanceadas se reporta **F1-macro** y la **matriz de confusión**, no solo accuracy.
6. Un resultado solo tiene sentido **comparado** con líneas base.

## Tarea: clasificador de estilo en español

**Objetivo:** aplicar el proceso completo de esta clase a un problema nuevo, en español.

1. Cada equipo construye un dataset de **al menos 200 frases en español** etiquetadas en dos estilos, por ejemplo **formal / informal** o **irónico / literal**, balanceado entre clases.
2. Divida el dataset en entrenamiento (70%), validación (15%) y prueba (15%).
3. Ajuste un modelo en español (`dccuchile/bert-base-spanish-wwm-cased`) reutilizando el código de este notebook.
4. Compare contra TF-IDF + regresión logística.
5. Entregue el notebook con: curvas de aprendizaje, `classification_report`, matriz de confusión, 10 errores analizados y una conclusión.

---
# Anexo. Cargar el modelo de respaldo

Si el entrenamiento no puede ejecutarse en clase (sin GPU o sin conexión), el docente comparte la carpeta `modelo_emociones_distilbert` generada en el Bloque 4.4. Súbala al entorno (o móntela desde Drive) y ejecute:

In [ ]:
# RUTA_MODELO = "/content/drive/MyDrive/modelo_emociones_distilbert"
modelo_respaldo = AutoModelForSequenceClassification.from_pretrained(RUTA_MODELO).to(DISPOSITIVO)
tokenizer = AutoTokenizer.from_pretrained(RUTA_MODELO)

predecir(modelo_respaldo, FRASES_DEMO)

Con el modelo de respaldo cargado, los Bloques 5 y 6 funcionan igual reemplazando `trainer.model` por `modelo_respaldo`. Para obtener `pred_test` sin `trainer`, cree uno solo para predecir:

```python
trainer = Trainer(model=modelo_respaldo, data_collator=collator, processing_class=tokenizer)
```

---
**Referencias**

- Saravia, E. et al. (2018). *CARER: Contextualized Affect Representations for Emotion Recognition*. EMNLP. (Dataset `dair-ai/emotion`)
- Sanh, V. et al. (2019). *DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter*. arXiv:1910.01108.
- Devlin, J. et al. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL.
- Hu, E. et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models*. arXiv:2106.09685.
- Tunstall, L., von Werra, L. y Wolf, T. (2022). *Natural Language Processing with Transformers*. O'Reilly.